In [1]:
import requests
from bs4 import BeautifulSoup
import urllib3
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import pandasql as ps
import os

# currentdate = datetime.now().strftime("%Y%m%d")
currentdate = '20250602'

In [5]:
t1t2 = pd.read_csv('CFP_CFO_Init.csv')
t3 = pd.read_csv(f'inner_join_{currentdate}.csv')
len(t3)

9178

In [6]:
category = pd.concat([t1t2['กลุ่ม'],t3['Industrials']],axis=0,ignore_index=True)
name = pd.concat([t1t2['ชื่อ'],t3['Name']],axis=0,ignore_index=True)
details = pd.concat([t1t2['รายละเอียด'],t3['Detail']],axis=0,ignore_index=True)
unit = pd.concat([t1t2['หน่วย'],t3['Unit']],axis=0,ignore_index=True)
ef = pd.concat([t1t2['ค่าแฟคเตอร์ (kgCO2e/หน่วย)'],t3['EF']],axis=0,ignore_index=True)
reference = pd.concat([t1t2['แหล่งข้อมูลอ้างอิง'],t3['Company_name']],axis=0,ignore_index=True)
last_updated = pd.concat([t1t2['วันที่อัพเดท'],t3['ApproveDate']],axis=0,ignore_index=True)

# new table
table = pd.DataFrame({
    'Category': category,
    'Name': name,
    'Detail': details,
    'Unit': unit,
    'Factor': ef,
    'Reference': reference,
    'Last_Updated': last_updated
})

In [7]:
len(table)

9989

In [9]:
import re

def convert_to_thai_date(date_string):
    """
    แปลงวันที่เป็นรูปแบบ MMM yyyy (พ.ศ.) เช่น Dec 2562, Jul 2565
    """
    if pd.isna(date_string) or not isinstance(date_string, str):
        return date_string
    
    # Dictionary สำหรับแปลงเดือน (3 ตัวอักษร)
    month_mapping = {
        'Jan': 'Jan', 'January': 'Jan',
        'Feb': 'Feb', 'February': 'Feb',
        'Mar': 'Mar', 'March': 'Mar',
        'Apr': 'Apr', 'April': 'Apr',
        'May': 'May',
        'Jun': 'Jun', 'June': 'Jun',
        'Jul': 'Jul', 'July': 'Jul',
        'Aug': 'Aug', 'August': 'Aug',
        'Sep': 'Sep', 'September': 'Sep',
        'Oct': 'Oct', 'October': 'Oct',
        'Nov': 'Nov', 'November': 'Nov',
        'Dec': 'Dec', 'December': 'Dec'
    }
    
    # Dictionary สำหรับแปลงจากตัวเลขเป็นเดือน
    number_to_month = {
        '01': 'Jan', '1': 'Jan',
        '02': 'Feb', '2': 'Feb',
        '03': 'Mar', '3': 'Mar',
        '04': 'Apr', '4': 'Apr',
        '05': 'May', '5': 'May',
        '06': 'Jun', '6': 'Jun',
        '07': 'Jul', '7': 'Jul',
        '08': 'Aug', '8': 'Aug',
        '09': 'Sep', '9': 'Sep',
        '10': 'Oct',
        '11': 'Nov',
        '12': 'Dec'
    }
    
    # กรณี 1: รูปแบบ "Dec 2019", "July 2022" - ไม่ต้องทำอะไร
    match = re.search(r'([A-Za-z]+)\s+(\d{4})', date_string)
    if match:
        month_name, year = match.groups()
        month_short = month_mapping.get(month_name, month_name[:3])
        
        # ถ้าเป็น format Dec 2019 อยู่แล้ว ไม่ต้องทำอะไร
        return f"{month_short} {year}"
    
    # กรณี 2: รูปแบบ "29/11/2565", "24/02/2568" - ลบ 543
    match = re.search(r'(\d{1,2})/(\d{1,2})/(\d{4})', date_string)
    if match:
        day, month, year = match.groups()
        year_int = int(year)
        month_name = number_to_month.get(month, 'Jan')
        
        # ลบ 543 เฉพาะปี พ.ศ. (> 2500)
        if year_int > 2500:
            thai_year = year_int - 543
        else:
            thai_year = year_int
            
        return f"{month_name} {thai_year}"
    
    # กรณี 3: มีแค่ปี เช่น "2565", "2568" - ลบ 543
    match = re.search(r'(\d{4})', date_string)
    if match:
        year = match.group(1)
        year_int = int(year)
        
        # ลบ 543 เฉพาะปี พ.ศ. (> 2500)
        if year_int > 2500:
            thai_year = year_int - 543
        else:
            thai_year = year_int
            
        return f"Jan {thai_year}"
    
    # ถ้าไม่ตรงกับรูปแบบไหน ส่งกลับเหมือนเดิม
    return date_string


table['Last_Updated'] = table['Last_Updated'].apply(convert_to_thai_date)

In [10]:
table

,Category,Name,Detail,Unit,Factor,Reference,Last_Updated
0,กลุ่มปิโตรเคมี,Acrylonitrile Butadiene Styrene (ABS),ผลิตจากกระบวนการอัลคิลเลชันของเบนซีนและเอทีลีน...,kg,4.1597,"Thai National LCI Database, TIIS-MTEC-NSTDA (w...",Dec 2019
1,กลุ่มปิโตรเคมี,General Purposed Polystyrene (GPPS),ผลิตจาก Styrene และ Ethylbenzene; LCIA method ...,kg,3.2281,"Thai National LCI Database, TIIS-MTEC-NSTDA (w...",Dec 2019
2,กลุ่มปิโตรเคมี,High Density Polyethylene (HDPE),ผลิตจาก Ethylene โดยมี 1-Butene และ Propylene ...,kg,6.7071,"Thai National LCI Database, TIIS-MTEC-NSTDA (w...",Dec 2019
3,กลุ่มปิโตรเคมี,High Impact Polystyrene (HIPS),ผลิตจาก Styrene และ Polybutadiene rubber; LCIA...,kg,3.6843,"Thai National LCI Database, TIIS-MTEC-NSTDA (w...",Dec 2019
4,กลุ่มปิโตรเคมี,Linear Low Density Polyethylene (LLDPE),ผลิตจากกระบวนการที่เป็น Solution phase และ Gas...,kg,2.1356,"Thai National LCI Database, TIIS-MTEC-NSTDA (w...",Jul 2022
...,...,...,...,...,...,...,...
9984,ปิโตรเคมี และเคมีภัณฑ์,ไฮโดรเจน,NaN,1 กิโลกรัม,1.06 kgCO2e,"บริษัท มาบตาพุดโอเลฟินส์ จำกัด, ระยอง",Nov 2022
9985,ปิโตรเคมี และเคมีภัณฑ์,ไฮโดรเจน,NaN,1 กิโลกรัม,1.94 kgCO2e,บริษัท พีทีที โกลบอล เคมิคอล จำกัด (มหาชน) (00...,Feb 2025
9986,ปิโตรเคมี และเคมีภัณฑ์,ไฮโดรเเว๊กซ์,NaN,1 กิโลกรัม,360 gCO2e,บริษัท พีทีที โกลบอล เคมิคอล จำกัด (มหาชน) (00...,Feb 2025
9987,อาหารสัตว์,ไฮโดรไลซ์ ขนสัตว์ปีกป่น สูตร 1,NaN,1 กิโลกรัม,806 gCO2e,บริษัท ศิริชัย เฟทเธอร์ อินดัสเทรียล จำกัด (สำ...,Mar 2024


In [9]:
for i in table['Category'].unique():
    print(i)

กลุ่มปิโตรเคมี
กลุ่มผลิตภัณฑ์จากก๊าซธรรมชาติ
กลุ่มพลังงาน: เชื้อเพลิงเหลว และเชื้อเพลิงแข็ง
กลุ่มไฟฟ้า
กลุ่มน้ำประปาและน้ำอุตสาหกรรม (Tap water)
กลุ่มการขนส่งโดยรถบรรทุก (Truck Transportations) และขนส่งประเภทอื่น ๆ (Others)
สิ่งทอ
กลุ่มอุตสาหกรรมยางธรรมชาติ (Natural rubber)
กลุ่มอุตสาหกรรมโรงเลื่อยและโรงอบไม้ยางพารา (Wood Processing : Para-wood)
ปาล์มน้ำมัน
กลุ่มอาหารสัตว์
กลุ่มผลิตภัณฑ์ทางการเกษตรและอาหาร
กลุ่มปศุสัตว์
กลุ่มผลิตภัณฑ์ที่ได้จากสัตว์และกลุ่มผลิตภัณฑ์ทางการเกษตร
กลุ่มเครื่องจักรกลทางการเกษตร
กลุ่มการจัดการมูลฝอยชุมชน และการปรับปรุงน้ำเสียชุมชน
กลุ่มเยื่อและกระดาษ
กลุ่มเคมีภัณฑ์ (Chemicals)
กลุ่มการฝังกลบขยะ
กลุ่มแก้วและกระจก
กลุ่มไหมหัตถกรรม (Sericulture)
Stationary Combustion
Mobile Combustion (On road)
Mobile Combustion (Off road), Diesel
Mobile Combustion (On road), Motor Gasoline 4 stroke
Mobile Combustion (On road), Motor Gasoline 2 stroke
Electricity, grid mix (ไฟฟ้า)
Refrigerants (สารทำความเย็น)
อื่นๆ
ปิโตรเคมี และเคมีภัณฑ์
ปิโตรเลี่ยม
ยานยนต์
หัตถกรรม และเครื่องปร

In [155]:
result = table.groupby('Category')['Name'].apply(lambda x: x.sample(n=min(2, len(x))))
for i in result:
    print(i)

อาหารหมูขุน  ไฮโกร  599  ขนาด  30  กิโลกรัม
อาหารเป็ดพันธุ์  ไฮโปรไวท์  546  ขนาด  1 ตัน
ไฟฟ้าแบบ grid mix ปี 2016-2018; LCIA method IPCC 2013 GWP 100a V1.03
Forestry
Agriculture
Gas/ Diesel Oil
Motor Gasoline - low mileage light duty vihicle vintage 1995 or later
Forestry
Agriculture
Industry
Agriculture
R-134a
R-134
Fuel oil A
Cob (CO2only)
บรรจุภัณฑ์กระดาษลูกฟูก 3 ชั้น ลอน C ชนิด RSC (KT-CSP-KT) (โรงงานราชบุรี)
บรรจุภัณฑ์กระดาษลูกฟูก 3 ชั้น ลอน C ชนิด RSC (KT-CA-KT) (โรงงานราชบุรี)
รถบรรทุกซีเมนต์ผง (ชนิดกล้วย) 18 ล้อ วิ่งสมบุกสมบัน 75% Loading
รถตู้บรรทุกพ่วง 18 ล้อ วิ่งแบบสมบุกสมบัน 50% Loading
การรวบรวมน้ำเสียชุมชนของเมืองขนาดใหญ่
ปุ๋ยหมักอินทรีย์ จากการจัดการมูลฝอยสด
กิ่งไม้ ต้นหญ้าจากสวน
ผ้า
น้ำปราศจากไอออน ที่ผลิตโดยเทคโนโลยี Reverse Osmosis
น้ำประปา-การประปานครหลวง
ปลานิล (เลี้ยงในบ่อดิน)
โคเนื้อมีชีวิต: ระยะเวลาขุน 6-12 เดือน
Styrene Monomer (SM)
Toluene
Ethane (อีเทน)
Natural Gas Liquid (ก๊าซธรรมชาติเหลว)
หน่อไม้ฝรั่ง (อินทรีย์)
กาแฟสารอราบิก้า (ค่าเฉลี่ย)
นมผึ้ง (Royal jel

In [11]:
currentdate = datetime.now().strftime("%Y%m%d")

table.to_json(f'combine_{currentdate}.json',orient='records',indent=4,force_ascii=False)
with open(f'combine_{currentdate}.json', 'r', encoding='utf-8') as f:
    json_str = f.read()
json_str = json_str.replace('\\/', '/')
json_str = json_str.replace('null','"NULL"')

with open(f'combine_{currentdate}.json', 'w', encoding='utf-8') as f:
    f.write(json_str)


destination_path = r'C:\Users\Nattapot\Desktop\thesis_ef\elasticsearch'
destination_file = os.path.join(destination_path, f'combine_{currentdate}.json')
with open(destination_file, 'w', encoding='utf-8') as f:
    f.write(json_str)